In [1]:
import os
print(os.getcwd())

/Users/lijiaying/Desktop/uk-ai-data-job-market-analysis/notebooks


In [2]:
import pandas as pd

jd = pd.read_csv("../data/raw/jd_full_text_20260810.csv")
meta = pd.read_csv("../data/processed/jobs_cleaned_20260808.csv")

df = jd.merge(meta, on="id", how="left")
print(df.shape)
print(df["jd_text"].str.len().describe())

(183, 15)
count      183.000000
mean      4506.786885
std       1964.707859
min        899.000000
25%       3058.000000
50%       4308.000000
75%       5790.500000
max      10446.000000
Name: jd_text, dtype: float64


In [3]:
SKILLS = {
    "cloud": ["aws", "s3", "ec2", "lambda", "sagemaker", "redshift",
              "athena", "azure", "gcp", "bigquery", "databricks",
              "snowflake", "terraform", "kubernetes", "docker"],
    "language": ["python", "r", "sql", "scala", "java", "c++",
                 "matlab", "sas", "julia", "fortran"],
    "library": ["pandas", "numpy", "scikit-learn", "pyspark", "spark",
                "airflow", "tensorflow", "pytorch", "keras", "git"],
    "bi": ["excel", "power bi", "tableau", "looker", "vba"],
    "method": ["machine learning", "deep learning", "nlp", "statistics",
               "a/b testing", "forecasting", "etl", "mlops",
               "numerical methods", "hpc", "parallel computing",
               "optimisation", "simulation", "rag", "vector database",
               "embeddings"],
}

all_skills = [s for group in SKILLS.values() for s in group]
print(len(all_skills))

df["jd_lower"] = df["jd_text"].str.lower()
print(df["jd_lower"].iloc[0][:200])

56
about us: lseg (london stock exchange group) is more than a diversified global financial markets infrastructure and data business. we are dedicated, open-access partners with a dedication to excellenc


In [4]:
naive = {}
for s in all_skills:
    naive[s] = df["jd_lower"].str.contains(s, regex=False).sum()

res = pd.Series(naive).sort_values(ascending=False)
print(res.head(20))

r                   183
excel               107
python               98
rag                  89
machine learning     77
scala                71
sql                  68
git                  65
statistics           44
aws                  35
azure                34
gcp                  25
mlops                23
pytorch              23
optimisation         23
forecasting          21
docker               21
power bi             20
tableau              18
databricks           17
dtype: int64


In [5]:
import re

def build_pattern(skill):
    """把技能词转成带词边界的正则"""
    return r"\b" + re.escape(skill) + r"\b"

for s in ["r", "rag", "scala", "excel", "c++", "a/b testing", "power bi"]:
    print(f"{s:15s} {build_pattern(s)}")

r               \br\b
rag             \brag\b
scala           \bscala\b
excel           \bexcel\b
c++             \bc\+\+\b
a/b testing     \ba/b\ testing\b
power bi        \bpower\ bi\b


In [14]:
counts = {}
for s in all_skills:
    p = build_pattern(s)
    counts[s] = df["jd_lower"].str.contains(p, regex=True).sum()

res2 = pd.Series(counts).sort_values(ascending=False)
print(res2.head(25))

python              96
machine learning    76
sql                 67
statistics          44
azure               34
aws                 32
gcp                 25
r                   23
pytorch             23
optimisation        23
mlops               23
docker              21
forecasting         21
excel               21
git                 20
power bi            20
databricks          17
tensorflow          17
tableau             17
kubernetes          15
deep learning       12
snowflake           11
a/b testing         11
pandas              11
scikit-learn        11
dtype: int64


In [15]:
def show_context(skill, n=8, width=60):
    """打印某技能词命中处的前后文"""
    p = build_pattern(skill)
    hits = df[df["jd_lower"].str.contains(p, regex=True)]
    print(f"=== {skill}  命中 {len(hits)} 条，抽 {min(n, len(hits))} 条 ===")
    for t in hits["jd_lower"].head(n):
        m = re.search(p, t)
        s = max(0, m.start() - width)
        e = min(len(t), m.end() + width)
        print("…" + t[s:e].replace("\n", " ") + "…")
    print()

for s in ["r", "excel", "git", "java"]:
    show_context(s)

=== r  命中 23 条，抽 8 条 ===
… and travel spend analysis and client data. utilize python, r, javascript, or tableau to build scalable, testable tools f…
…iency with python or other scripting/programming languages (r, matlab, etc) ability to bring clarity to complex domains, …
… analytical tools and programming languages such as python, r, and sql. a keen interest in machine learning techniques, f…
…and emblem. query and engineer large datasets (e.g., python/r/sql/pyspark) on modern platforms (e.g., azure databricks). …
…facing environments · advanced proficiency in python and/or r, with experience operationalising data science solutions an…
…sis is a plus. technical skills: proficiency in sql, python/r, and data visualization tools (e.g., power bi, tableau) a m…
…lizing tabular data using a combination of sql, polars, and r basic software development skills and experience with bash,…
…ence including python and associated packages, e-views, and r. why join our team? we support our people to

In [13]:
def build_pattern(skill):
    """带词边界的正则；对已知歧义词做特殊处理"""
    special = {
        # R：排除 r&d
        "r": r"\br\b(?!\s*&\s*d\b)",
        # excel：排除动词用法 "excel at/in/if/when"
        "excel": r"\bexcel\b(?!\s+(?:at|in|if|when)\b)",
        # git：纳入 github、gitlab
        "git": r"\bgit(?:hub|lab)?\b",
        # sql：纳入 postgresql、mysql、nosql 等变体
        "sql": r"\b(?:my|postgre|no|t-|pl/)?sql\b",
    }
    if skill in special:
        return special[skill]
    return r"\b" + re.escape(skill) + r"\b"

for s in ["r", "excel", "git", "sql", "python"]:
    print(f"{s:10s} {build_pattern(s)}")

r          \br\b(?!\s*&\s*d\b)
excel      \bexcel\b(?!\s+(?:at|in|if|when)\b)
git        \bgit(?:hub|lab)?\b
sql        \b(?:my|postgre|no|t-|pl/)?sql\b
python     \bpython\b


In [19]:
df["title_lower"] = df["title"].str.lower()

SENIORITY = [
    ("graduate", r"\b(graduate|intern|internship|placement|trainee|apprentice)\b"),
    ("junior",   r"\b(junior|entry.level|jr\.?)\b"),
    ("senior",   r"\b(senior|snr\.?|sr\.?)\b"),
    ("lead",     r"\b(lead|principal|staff|head\s+of|director|chief)\b"),
]

def get_seniority(t):
    for label, pat in SENIORITY:
        if re.search(pat, t):
            return label
    return "unspecified"

df["seniority"] = df["title_lower"].apply(get_seniority)
print(df["seniority"].value_counts())

seniority
unspecified    104
senior          41
lead            25
graduate        13
Name: count, dtype: int64


In [17]:
for lab in ["graduate", "junior", "lead"]:
    print(f"=== {lab} ===")
    print(df[df["seniority"] == lab]["title"].head(10).to_string(index=False))
    print()

=== graduate ===
                           Graduate Water Modeller
                         Graduate Business Analyst
Graduate Geospatial (GIS) Consultant - Communit...
                           Graduate Data Scientist
                      Analytics Graduate Programme
                         Data and Analytics Intern
           Data Analyst Intern - Financial Markets
                             Graduate Data Analyst
Data Analyst Placement Programme No Experience ...
                             Graduate Data Analyst

=== junior ===
Series([], )

=== lead ===
                          Principal Data Scientist
                  Product Manager - Reference Data
        Staff Software Engineer - Machine Learning
                          Principal Data Scientist
                       Lead Product Manager (Data)
                    Lead Machine Learning Engineer
                               Lead Data Scientist
                   Staff Machine Learning Engineer
Principal Machine Lear

In [18]:
print(df[df["title_lower"].str.contains("junior|jr|entry", regex=True)]["title"].to_string(index=False))
print("---")
print(df[df["title_lower"].str.contains("manager", regex=True)]["title"].to_string(index=False))

Series([], )
---
       Senior Data Science Manager
  Product Manager - Reference Data
       Lead Product Manager (Data)
            Media Insights Manager
          Senior Ecommerce Manager
      Product Data Science Manager
         Principal Product Manager
                   Project Manager
Staff Data Science Product Manager
          Senior Programme Manager
          Senior Programme Manager


In [20]:
meta_all = pd.read_csv("../data/processed/jobs_cleaned_20260808.csv")
t = meta_all["title"].str.lower()

for pat in ["junior", "jr", "entry", "graduate", "trainee", "placement"]:
    n = t.str.contains(r"\b" + pat + r"\b", regex=True).sum()
    print(f"{pat:12s} {n:>4d}  ({n/len(meta_all)*100:.1f}%)")

junior         15  (1.1%)
jr              0  (0.0%)
entry           2  (0.1%)
graduate       42  (3.1%)
trainee        43  (3.2%)
placement      15  (1.1%)


In [21]:
for s in all_skills:
    df["has_" + s] = df["jd_lower"].str.contains(build_pattern(s), regex=True)

skill_cols = ["has_" + s for s in all_skills]

grouped = df.groupby("seniority")[skill_cols].mean().T
grouped.columns = [f"{c}(n={(df['seniority']==c).sum()})" for c in grouped.columns]
grouped["overall"] = df[skill_cols].mean()

top = grouped.sort_values("overall", ascending=False).head(20)
print((top * 100).round(1).to_string())

                      graduate(n=13)  lead(n=25)  senior(n=41)  unspecified(n=104)  overall
has_python                      30.8        52.0          63.4                51.0     52.5
has_machine learning            15.4        52.0          43.9                41.3     41.5
has_sql                         30.8        20.0          48.8                36.5     36.6
has_statistics                  15.4        16.0          29.3                25.0     24.0
has_azure                        7.7        12.0          26.8                18.3     18.6
has_aws                          7.7        16.0          22.0                17.3     17.5
has_gcp                          7.7        16.0          19.5                11.5     13.7
has_r                           15.4        12.0          17.1                10.6     12.6
has_pytorch                      7.7        20.0          17.1                 9.6     12.6
has_optimisation                 7.7        16.0          14.6                11

In [22]:
df.to_csv("../data/processed/jobs_with_skills_20260810.csv", index=False)
grouped.to_csv("../data/processed/skill_by_seniority_20260810.csv")
print(df.shape)

(183, 74)
